# Getting Started: IMDB Sentiment Classification

This notebook provides a quick introduction to the IMDB sentiment classification benchmark.

**What you'll learn:**
- How to load the IMDB dataset in Colab
- Run your first experiment (TF-IDF + Logistic Regression)
- Interpret the results

**Time required:** ~5-10 minutes

**Note**: This notebook is designed to run in Google Colab.

## 1. Setup

First, let's set up the environment.

In [ ]:
# If running in Colab, clone the repository
import os

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('Embedding_models_with_Classification'):
        !git clone https://github.com/rylex27-z/Embedding_models_with_Classification.git
    %cd Embedding_models_with_Classification
else:
    print("Not running in Colab. Make sure you're in the project directory.")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

## 2. Upload Dataset

You need to upload the IMDB dataset. See `data/README.md` for download instructions.

**Option 1**: Upload `aclImdb.zip` to `/content/`

In [ ]:
# Upload dataset (uncomment if needed)
# from google.colab import files
# uploaded = files.upload()  # Select aclImdb.zip

**Option 2**: Mount Google Drive if dataset is there

In [ ]:
# Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/data/aclImdb.zip'

# Or if uploaded to /content/
DATA_PATH = '/content/aclImdb.zip'

## 3. Load Data

Let's load a small sample for quick testing.

In [ ]:
import sys
sys.path.insert(0, 'src')

from src.data_loader import load_and_preprocess_imdb
import numpy as np

# Load small sample for quick testing (1000 per class)
print("Loading IMDB dataset...")
train_texts, train_labels, test_texts, test_labels = load_and_preprocess_imdb(
    zip_path=DATA_PATH,
    sample_size=1000,  # Small sample for quick testing
    preprocess=True,
    random_seed=42
)

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print(f"\nDataset loaded:")
print(f"  Training samples: {len(train_texts)}")
print(f"  Test samples: {len(test_texts)}")
print(f"  Positive labels: {sum(train_labels == 1)}")
print(f"  Negative labels: {sum(train_labels == 0)}")

## 4. Example: TF-IDF + Logistic Regression

Let's run a simple baseline experiment.

In [ ]:
from src.embeddings import TfidfEmbedding
from src.models import get_model
from src.evaluation import run_full_evaluation

# Create embedding
print("Creating TF-IDF embedding...")
embedding = TfidfEmbedding(max_features=5000, ngram_range=(1, 2))

# Create model
print("Creating Logistic Regression model...")
model = get_model('logreg', config={'max_iter': 1000, 'random_state': 42})

# Run evaluation (with reduced CV for speed)
print("\nRunning evaluation with 3-fold CV...")
results = run_full_evaluation(
    embedding_obj=embedding,
    model_obj=model,
    X_train=train_texts,
    y_train=train_labels,
    X_test=test_texts,
    y_test=test_labels,
    embedding_type='tfidf',
    embedding_variant='max_features_5000',
    model_type='logreg',
    cv_config={'n_splits': 3, 'n_repeats': 1, 'random_seeds': [42]},
    verbose=True
)

## 5. View Results

In [ ]:
import pandas as pd

# Cross-validation results
print("\n" + "="*80)
print("Cross-Validation Results (per fold)")
print("="*80)
print(results['cv_results'])

# Summary statistics
print("\n" + "="*80)
print("Summary Statistics")
print("="*80)
summary_df = pd.DataFrame([results['cv_summary']])
print(summary_df.T)

# Test set performance
print("\n" + "="*80)
print("Test Set Performance")
print("="*80)
test_df = pd.DataFrame([results['test_metrics']])
print(test_df.T)

## 6. Understanding the Results

**Cross-Validation Results**: Performance on different folds of the training data
- Shows stability and variance across different splits
- Mean ± Std gives robust estimate

**Test Set Performance**: Final performance on held-out test data
- This is what you would report
- Should be similar to CV performance (if not, may indicate issues)

**Metrics**:
- **Accuracy**: Overall correctness (good for balanced datasets)
- **F1**: Harmonic mean of precision and recall (good overall metric)
- **Precision**: Of predicted positives, how many are correct?
- **Recall**: Of actual positives, how many did we find?

## 7. Try Another Method (Optional)

Uncomment to try Word2Vec + Random Forest:

In [ ]:
# from src.embeddings import Word2VecEmbedding

# # Word2Vec Skip-gram
# embedding_w2v = Word2VecEmbedding(vector_size=100, window=5, sg=1, epochs=5)

# # Random Forest
# model_rf = get_model('randomforest', config={'n_estimators': 50, 'random_state': 42})

# # Evaluate
# results_w2v = run_full_evaluation(
#     embedding_obj=embedding_w2v,
#     model_obj=model_rf,
#     X_train=train_texts,
#     y_train=train_labels,
#     X_test=test_texts,
#     y_test=test_labels,
#     embedding_type='word2vec',
#     embedding_variant='skipgram',
#     model_type='randomforest',
#     cv_config={'n_splits': 3, 'n_repeats': 1, 'random_seeds': [42]},
#     verbose=True
# )

## Next Steps

1. **Run full experiments**: Remove `sample_size` parameter to use full dataset
2. **Try BERT**: See `full_benchmark.ipynb` for BERT examples
3. **Use CLI scripts**: For systematic experiments
   ```bash
   python scripts/run_experiment.py --embedding tfidf --model logreg
   ```
4. **Read documentation**:
   - `docs/theory_embeddings.md` - Learn about embeddings
   - `docs/theory_models.md` - Learn about models
   - `docs/experiment_protocol.md` - Full experimental methodology

Happy experimenting! 🚀